In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import os
import copy
from sklearn.metrics import accuracy_score

In [6]:
import sys
sys.path.append(os.path.abspath("../data"))
from models.gcn_model import GCN

Check Device (CPU/GPU)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


Define the GCN Model (CORE MODEL)

Training Function (Reusable for Any Hospital)

In [8]:
def train_local_model(graph_path, epochs=100, lr=0.01):
    graph = torch.load(graph_path, weights_only=False).to(device)

    model = GCN(
        input_dim=graph.num_node_features,
        hidden_dim=32,
        output_dim=2
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(graph.x, graph.edge_index)
        loss = criterion(out, graph.y)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"[{graph_path}] Epoch {epoch:03d} | Loss: {loss:.4f}")

    return model

In [9]:
# Evaluation Function
def evaluate_model(model, graph_path):
    graph = torch.load(graph_path, weights_only=False).to(device)
    model.eval()

    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        preds = logits.argmax(dim=1).cpu()
        labels = graph.y.cpu()

    return accuracy_score(labels, preds)

Train Models at Each Hospital

In [10]:
model_A = train_local_model("../data/graph/graph_A.pt")
model_B = train_local_model("../data/graph/graph_B.pt")
model_C = train_local_model("../data/graph/graph_C.pt")

[../data/graph/graph_A.pt] Epoch 000 | Loss: 0.4801
[../data/graph/graph_A.pt] Epoch 020 | Loss: 0.0665
[../data/graph/graph_A.pt] Epoch 040 | Loss: 0.0415
[../data/graph/graph_A.pt] Epoch 060 | Loss: 0.0396
[../data/graph/graph_A.pt] Epoch 080 | Loss: 0.0359
[../data/graph/graph_B.pt] Epoch 000 | Loss: 0.4233
[../data/graph/graph_B.pt] Epoch 020 | Loss: 0.0797
[../data/graph/graph_B.pt] Epoch 040 | Loss: 0.0612
[../data/graph/graph_B.pt] Epoch 060 | Loss: 0.0485
[../data/graph/graph_B.pt] Epoch 080 | Loss: 0.0539
[../data/graph/graph_C.pt] Epoch 000 | Loss: 0.8641
[../data/graph/graph_C.pt] Epoch 020 | Loss: 0.0283
[../data/graph/graph_C.pt] Epoch 040 | Loss: 0.0154
[../data/graph/graph_C.pt] Epoch 060 | Loss: 0.0104
[../data/graph/graph_C.pt] Epoch 080 | Loss: 0.0092


In [11]:
print("Hospital A Accuracy (Clean):", evaluate_model(model_A, "../data/graph/graph_A.pt"))
print("Hospital B Accuracy (Clean):", evaluate_model(model_B, "../data/graph/graph_B.pt"))
print("Hospital C Accuracy (Clean):", evaluate_model(model_C, "../data/graph/graph_C.pt"))

Hospital A Accuracy (Clean): 0.990625
Hospital B Accuracy (Clean): 0.9829545454545454
Hospital C Accuracy (Clean): 1.0


In [14]:
# Save ONLY LDP-Protected Models
os.makedirs("../data/models", exist_ok=True)

torch.save(model_A.state_dict(), "../data/models/model_A.pth")
torch.save(model_B.state_dict(), "../data/models/model_B.pth")
torch.save(model_C.state_dict(), "../data/models/model_C.pth")

print("local models saved successfully")

local models saved successfully
